# EXP-ARCH-002 H1 Dominance Terminal: Prospective Kaggle Execution
## Hardware Target: Kaggle GPU T4 × 2 (Multi-vCPU Execution)

This notebook executes the formal prospective re-execution of **EXP-ARCH-002_H1_DOMINANCE_TERMINAL**.
The original execution had an instrumentation flaw (state pollution during integrity checks) and was invalidated.
This run uses the corrected runner (**P-RE-RUN**) with independent twin baseline verification.


In [ ]:
# Section 1: RUN IDENTITY
EXPERIMENT_ID = "EXP-ARCH-002_H1_DOMINANCE_TERMINAL"
PREREGISTRATION_DOC = "02_PREREGISTRATION.md"
COHORT_LO = 320000
COHORT_HI = 320499
TOTAL_SEEDS = 500
GAMES_PER_SEED = 2
INTENDED_GAMES = 1000
DESIGN = "Common Random Numbers (CRN): Game a (H1 seat 0), Game b (H1 seat 1)"

print("=" * 70)
print(f"EXPERIMENT IDENTITY: {EXPERIMENT_ID}")
print(f"GOVERNING PROTOCOL:  {PREREGISTRATION_DOC}")
print(f"AUTHORIZED COHORT:   Seeds {COHORT_LO} to {COHORT_HI} inclusive (N = {TOTAL_SEEDS} seeds)")
print(f"TOTAL GAMES:         {INTENDED_GAMES} games ({DESIGN})")
print("=" * 70)


## Section 2: KAGGLE ENVIRONMENT
Inspect compute runtime, OS, Python version, CPU threads, RAM, and GPU status.


In [ ]:
# Section 2: KAGGLE ENVIRONMENT
import os
import sys
import platform
import subprocess
import shutil

print(f"Python Version: {sys.version.split()[0]}")
print(f"Platform:       {platform.platform()}")
print(f"OS Name:        {os.name}")
print(f"CPU Count:      {os.cpu_count()} logical cores")

# RAM Detection
try:
    with open("/proc/meminfo", "r") as f:
        for line in f:
            if "MemTotal" in line:
                print(f"System RAM:     {line.strip()}")
                break
except Exception:
    pass

# GPU Detection
gpu_available = False
try:
    import torch
    if torch.cuda.is_available():
        gpu_available = True
        device_count = torch.cuda.device_count()
        print(f"PyTorch CUDA:   Available ({device_count} device(s))")
        for i in range(device_count):
            props = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
    else:
        print("PyTorch CUDA:   Not available")
except ImportError:
    print("PyTorch:        Not installed")

# nvidia-smi
if shutil.which("nvidia-smi"):
    try:
        smi_out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], text=True)
        print(f"nvidia-smi:     {smi_out.strip()}")
    except Exception as e:
        print(f"nvidia-smi error: {e}")

# Explicit Simulator GPU Utilization Declaration
print("\n" + "-" * 70)
print("ACCELERATOR RECORD:")
print("Hardware Target: Kaggle GPU T4 x 2")
print("Simulator GPU Status: GPU available but not used by simulator;")
print("                     execution performed on Kaggle CPU resources within T4x2 runtime.")
print("-" * 70)
print(f"Current Directory: {os.getcwd()}")


## Section 3: SOURCE INSTALLATION
Locate the source bundle, establish `/kaggle/working/Kagriculture` as canonical root, and configure `sys.path`.


In [ ]:
# Section 3: SOURCE INSTALLATION
from pathlib import Path
import tarfile
import shutil

CANONICAL_ROOT = Path("/kaggle/working/Kagriculture").resolve()
FALLBACK_ROOT = Path(os.getcwd()).resolve()

# 1. Determine or setup canonical root
if CANONICAL_ROOT.exists() and (CANONICAL_ROOT / "candidates" / "compiled_v54_plus_residual.py").exists():
    REPO_ROOT = CANONICAL_ROOT
    print(f"Found existing repository at canonical root: {REPO_ROOT}")
else:
    # Look for source bundle in common locations
    search_dirs = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    tar_found = None
    for d in search_dirs:
        if d.exists():
            for p in d.rglob("kaggle_arch002_source.tar.gz"):
                tar_found = p
                break
        if tar_found:
            break

    if tar_found:
        print(f"Extracting source archive from: {tar_found} -> {CANONICAL_ROOT.parent}")
        CANONICAL_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(tar_found, "r:gz") as tar:
            tar.extractall(CANONICAL_ROOT.parent)
        REPO_ROOT = CANONICAL_ROOT
    elif (FALLBACK_ROOT / "candidates" / "compiled_v54_plus_residual.py").exists():
        # Local/inline execution fallback
        REPO_ROOT = FALLBACK_ROOT
        print(f"Using local repository root: {REPO_ROOT}")
    else:
        raise RuntimeError(f"Could not locate Kagriculture source repository or archive!")

# Switch directory to REPO_ROOT
os.chdir(REPO_ROOT)
print(f"Working Directory set to: {Path.cwd()}")
if REPO_ROOT == CANONICAL_ROOT:
    assert Path.cwd() == CANONICAL_ROOT, f"CWD {Path.cwd()} != {CANONICAL_ROOT}"

# Configure Python path
PORT_DIR = str(REPO_ROOT / "experiments" / "EXP-DEV-006_HORIZON_KILLTEST" / "port")
H1_DIR = str(REPO_ROOT / "experiments" / "EXP-ARCH-002_H1_DOMINANCE_TERMINAL")

for p in (PORT_DIR, H1_DIR, str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Configured sys.path with {PORT_DIR} and {H1_DIR}")


## Section 4: HASH VERIFICATION (G-HASH)
Verify SHA-256 digests of all frozen governing documents, baseline policy, simulator engine, unit model, and H1 source code.
Any mismatch causes immediate hard abort.


In [ ]:
# Section 4: HASH VERIFICATION
import hashlib

def sha256_of_file(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

FROZEN_HASHES = {
    # Baseline & Extract
    "candidates/compiled_v54_plus_residual.py": "18402ec6fa3709b9b5ef90dd9c8d7b9c0e44aea55ab163e44a80a4adc0cb7eeb",
    "extracted_agents/dmitrii_lb2800/unit_model.py": "c03e8c5cd10466c229ee0a44dece9a3943c10bff135d96a7822ffe75d6922724",
    # Governing Docs
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/01_ISOLATION_AUDIT.md": "0381a93eee7cc3d51833439ce868d51923dc9a2ff0cca68f9d692e5f81c03de4",
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/02_PREREGISTRATION.md": "508f1dff91f1bfd34186b1eeb6e899c97f87b99fff3841513abe3a88dc4a71b5",
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/07_REEXECUTION_ELIGIBILITY_AUDIT.md": "06870a49e75790026823dda7102c44f1b7fdbec6965600b3362bf31179f60ade",
    # Runner
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/run_arch002_h1_reexecution.py": "e6f830c8847985f834dd3c498537e07f20fb006494e0596773e4f09d86b6a405",
    # H1 Modules
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/h1_const.py": "080d0992ff1235dddbcf1be5bd0d5eb7a9c0eaf533ef0f725941f39ba226479c",
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/h1_units.py": "9100029a1eac43d8cac527f79b33c865233435f52fd40f18e931278468fc8033",
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/h1_baseline.py": "20e7728f196228a022e762b7490b2081085f4ed639b48adecc810c1d2a0e4e58",
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/h1_dominance.py": "37d36383f32f68d7a437331a709d07bcbe0ac2a88cf9265fa9ea36e26c623a23",
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/h1_search.py": "1d476f7d7292f5f8af13ced0133b99f0dbb0e10499c64a5bd850ae394bbdaf9b",
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/h1_shadow.py": "8857a671e6d510ddb44957126b6839584d4264fc8d20f56ab94d7f9ea046055b",
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/h1_telemetry.py": "dd7c9aa3a45fc0dea0b020d56b7c9368d071e9ae9f0e031e0026d228a4c6b31c",
    "experiments/EXP-ARCH-002_H1_DOMINANCE_TERMINAL/h1_agent.py": "06aa31c72ea8c9740be2568810b7ade17edee1827e779b88cceddda70a40f4ae",
}

print("Executing G-HASH Pre-Execution Integrity Verification...")
all_passed = True
for relpath, expected_sha in FROZEN_HASHES.items():
    fpath = REPO_ROOT / relpath
    if not fpath.exists():
        print(f"FAILED: Missing file {relpath}")
        all_passed = False
        continue
    actual_sha = sha256_of_file(str(fpath))
    if actual_sha != expected_sha:
        print(f"FAILED: Hash mismatch on {relpath}")
        print(f"  Expected: {expected_sha}")
        print(f"  Observed: {actual_sha}")
        all_passed = False
    else:
        print(f"  [OK] {relpath}")

assert all_passed, "CRITICAL G-HASH INTEGRITY FAILURE! STOPPING."
print("\nG-HASH SUCCESS: All 14 frozen artifacts verified bit-identical.")


## Section 5: DEPENDENCY VERIFICATION
Verify importability of Python modules and assert complete network isolation.


In [ ]:
# Section 5: DEPENDENCY VERIFICATION
print("Verifying core imports...")
import kagsim_py.api as papi
import h1_const
import h1_units
import h1_baseline
import h1_dominance
import h1_search
import h1_shadow
import h1_telemetry
import h1_agent
import run_arch002_h1_reexecution as runner

# Assert no forbidden network references in runner
with open(REPO_ROOT / "experiments" / "EXP-ARCH-002_H1_DOMINANCE_TERMINAL" / "run_arch002_h1_reexecution.py", "r") as f:
    runner_code = f.read()

for forbidden in ("kaggle", "urllib.request", "requests", "http.client", "socket."):
    assert forbidden not in runner_code, f"Forbidden network keyword '{forbidden}' detected in runner!"

print("DEPENDENCY VERIFICATION PASSED: All local modules imported successfully with zero network dependencies.")


## Section 6: SIMULATOR VERIFICATION
Verify the Python simulator port engine identity, engine version, and smoke-test 10 steps on non-prospective consumed fixture.


In [ ]:
# Section 6: SIMULATOR VERIFICATION
print(f"kagsim_py location:       {papi.__file__}")
print(f"kagsim_py port version:   {papi.__version__}")
print(f"kagsim_py engine version: {papi.ENGINE_VERSION}")

assert papi.ENGINE_VERSION == "1.32.7", f"Wrong engine version: {papi.ENGINE_VERSION}"

# Smoke test on consumed fixture seed 248000
test_game = papi.Game(248000)
for step in range(10):
    obs0 = test_game.observe(0)
    obs1 = test_game.observe(1)
    test_game.step({}, {})

print(f"Simulator smoke test: successfully stepped 10 turns on fixture seed 248000.")
print("SIMULATOR VERIFICATION PASSED.")


## Section 7: RUNNER INTEGRITY (Phase G Gate)
Verify quarantine enforcement against Final Holdout (`214200–214249`), DEV-007 (`310000–310499`), and out-of-cohort seeds.
Execute Phase G validation test suite.


In [ ]:
# Section 7: RUNNER INTEGRITY
import pytest

# 1. Test quarantine aborts
print("Testing quarantine gates...")
for holdout_seed in (214200, 214225, 214249):
    try:
        runner.assert_quarantine(holdout_seed, allow_consumed_validation=False)
        raise AssertionError(f"Quarantine failed to abort on holdout seed {holdout_seed}!")
    except RuntimeError:
        pass

for dev007_seed in (310000, 310250, 310499):
    try:
        runner.assert_quarantine(dev007_seed, allow_consumed_validation=False)
        raise AssertionError(f"Quarantine failed to abort on DEV-007 seed {dev007_seed}!")
    except RuntimeError:
        pass

for ooc_seed in (100, 319999, 320500):
    try:
        runner.assert_quarantine(ooc_seed, allow_consumed_validation=False)
        raise AssertionError(f"Quarantine failed to abort on out-of-cohort seed {ooc_seed}!")
    except RuntimeError:
        pass

print("Quarantine hard-aborts verified: Holdout, DEV-007, and Out-of-Cohort strictly blocked.")

# 2. Run Phase G Validation Suite
val_test_path = str(REPO_ROOT / "experiments" / "EXP-ARCH-002_H1_DOMINANCE_TERMINAL" / "tests" / "test_h1_reexecution_validation.py")
print(f"\nRunning Phase G test suite: {val_test_path}")
ret_code = pytest.main(["-q", val_test_path])
assert ret_code == 0, f"Phase G validation tests failed with exit code {ret_code}!"
print("RUNNER INTEGRITY PASSED: All 12 validation tests passed 100%.")


## Section 8: PRE-ENTRY PURITY TEST
Directly prove that:
1. No policy instance is called twice per turn (zero double-calls).
2. H1 pre-E actions (steps 0 to 711) match unpolluted baseline actions with 100% bit-identity.
3. Abstained games produce `delta == 0.00`.


In [ ]:
# Section 8: PRE-ENTRY PURITY TEST
import copy

print("Executing P-PURE pre-E zero double-call verification on consumed seeds 248000, 248001, 248002...")
for seed in (248000, 248001, 248002):
    cfg = runner._get_game_config()
    b_h1 = runner.load_baseline()
    b_ctrl = runner.load_baseline()
    b_twin = runner.load_baseline()

    counts_h1 = {}
    counts_ctrl = {}
    counts_twin = {}

    orig_h1 = b_h1.call
    orig_ctrl = b_ctrl.call
    orig_twin = b_twin.call

    b_h1.call = lambda obs, c: counts_h1.setdefault(obs.get("step", 0), 0) or orig_h1(obs, c) or orig_h1(obs, c) # wrapper
    def trace(target_dict, fn):
        def wrapped(obs, c):
            s = obs.get("step", 0)
            target_dict[s] = target_dict.get(s, 0) + 1
            return fn(obs, c)
        return wrapped

    b_h1.call = trace(counts_h1, orig_h1)
    b_ctrl.call = trace(counts_ctrl, orig_ctrl)
    b_twin.call = trace(counts_twin, orig_twin)

    a_h1 = runner.build_agent(b_h1, shadow_only=False)
    g = papi.Game(seed)

    while not g.done:
        t = g.step_count
        o0 = g.observe(0)
        o1 = g.observe(1)
        act0 = a_h1(copy.deepcopy(o0), copy.deepcopy(cfg))
        act1 = b_ctrl.call(copy.deepcopy(o1), copy.deepcopy(cfg))

        if t < runner.ENTRY_STEP:
            act_twin = b_twin.call(copy.deepcopy(o0), copy.deepcopy(cfg))
            assert runner._actions_equal(act0, act_twin), f"Seed {seed} step {t}: action drift!"

        g.step(act0, act1)

    for s in range(runner.ENTRY_STEP):
        assert counts_h1.get(s, 0) == 1, f"Seed {seed} step {s}: b_h1 called {counts_h1.get(s)} times!"
        assert counts_twin.get(s, 0) == 1, f"Seed {seed} step {s}: b_twin called {counts_twin.get(s)} times!"
        assert counts_ctrl.get(s, 0) == 1, f"Seed {seed} step {s}: b_ctrl called {counts_ctrl.get(s)} times!"

    print(f"  [OK] Seed {seed}: zero double-calls, 100% pre-E bit-identity.")

print("\nPRE-ENTRY PURITY VERIFIED: P-PURE defect is completely absent.")


## Section 9: TERMINAL-WINDOW TEST
Verify terminal micro-transitions (steps 710 to 718) on consumed seed 248000.


In [ ]:
# Section 9: TERMINAL-WINDOW TEST
print("Testing terminal window micro-transitions on consumed seed 248000...")
res = runner.run_seed_crn(248000, allow_consumed_validation=True)
ga = res["game_a"]
gb = res["game_b"]

print(f"Seed 248000 results: paired_delta={res['delta_paired']:.2f}")
print(f"  Game a: delta={ga['delta']:.2f}, reason={ga['reason']}, route={ga['route']}, pure_pre_e={ga['pure_pre_e_pass']}")
print(f"  Game b: delta={gb['delta']:.2f}, reason={gb['reason']}, route={gb['route']}, pure_pre_e={gb['pure_pre_e_pass']}")

assert ga["pure_pre_e_pass"] is True
assert gb["pure_pre_e_pass"] is True
assert res["delta_paired"] == 0.0

print("TERMINAL WINDOW TRANSITIONS PASSED.")


## Section 10: MEMORY / WORKER CONFIGURATION
Configure conservative multiprocessing parameters, memory management hooks, and continuous checkpoint paths.


In [ ]:
# Section 10: MEMORY / WORKER CONFIGURATION
import gc

# 1. Conservative worker count (matching Kaggle 4 vCPUs)
available_cpus = os.cpu_count() or 4
WORKERS = min(4, max(1, available_cpus))
print(f"Worker Configuration: {WORKERS} parallel processes (Available cores: {available_cpus})")

# 2. Configure Output Directory
OUTPUT_DIR = REPO_ROOT / "experiments" / "EXP-ARCH-002_H1_DOMINANCE_TERMINAL" / "kaggle_execution"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint Output Directory: {OUTPUT_DIR}")

# Redirect runner checkpoint targets
runner.RAW_RESULTS_FILE = str(OUTPUT_DIR / "RAW_RESULTS.json")
runner.PAIRED_DELTAS_FILE = str(OUTPUT_DIR / "PAIRED_DELTAS.csv")
runner.TELEMETRY_EVENTS_FILE = str(OUTPUT_DIR / "TELEMETRY_EVENTS.csv")
runner.MANIFEST_FILE = str(OUTPUT_DIR / "EXECUTION_MANIFEST.json")
runner.LOG_FILE = str(OUTPUT_DIR / "EXECUTION_LOG.txt")

print(f"Logging to: {runner.LOG_FILE}")
print(f"Raw results to: {runner.RAW_RESULTS_FILE}")
print("ALL PREFLIGHT GATES SATISFIED. READY FOR PROSPECTIVE EXECUTION.")


## Section 11: PROSPECTIVE EXECUTION (500 Seeds / 1,000 Games)
**THIS IS THE REAL EXECUTION.**

- **Cohort**: Seeds `320000` to `320499` inclusive (N = 500 seeds)
- **Games**: Exactly 1,000 games under Common Random Numbers (CRN)
- **Checkpointing**: Continuous atomic serialization every 10 seeds
- **Resume Support**: Automatically detects and resumes completed seeds if interrupted


In [ ]:
# Section 11: PROSPECTIVE EXECUTION
import time
import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed

def execute_with_memory_discipline(workers: int = 4):
    start_time_iso = datetime.datetime.now(datetime.timezone.utc).isoformat()
    results_map, already_done = runner.load_existing_results()
    
    if already_done > 0:
        runner.log(f"Resuming run: found {already_done} already completed seeds.")
    
    seeds_to_run = [s for s in range(runner.AUTHORIZED_COHORT_LO, runner.AUTHORIZED_COHORT_HI + 1) if s not in results_map]
    runner.log(f"Seeds remaining to execute: {len(seeds_to_run)} / {runner.AUTHORIZED_SEEDS_COUNT}")
    
    if not seeds_to_run:
        runner.log("All 500 seeds already executed.")
        runner.save_results_checkpoint(results_map, start_time_iso, status="COMPLETED")
        return results_map

    t0 = time.perf_counter()
    completed_count = already_done

    # Run in batches to prevent ProcessPoolExecutor task queue memory accumulation
    BATCH_SIZE = 20
    for b_start in range(0, len(seeds_to_run), BATCH_SIZE):
        batch = seeds_to_run[b_start : b_start + BATCH_SIZE]
        with ProcessPoolExecutor(max_workers=workers) as pool:
            future_to_seed = {pool.submit(runner.run_seed_crn, s, False): s for s in batch}
            for future in as_completed(future_to_seed):
                seed = future_to_seed[future]
                try:
                    seed_res = future.result()
                    results_map[seed] = seed_res
                    completed_count += 1
                except Exception as exc:
                    runner.log(f"FATAL ERROR on seed {seed}: {exc}")
                    runner.save_results_checkpoint(results_map, start_time_iso, status="FAILED_EXCEPTION")
                    raise

                if completed_count % 10 == 0 or completed_count == runner.AUTHORIZED_SEEDS_COUNT:
                    elapsed = time.perf_counter() - t0
                    rate = (completed_count - already_done) / max(elapsed, 0.001)
                    runner.log(f"Progress: {completed_count}/{runner.AUTHORIZED_SEEDS_COUNT} seeds "
                               f"({completed_count * 2}/{runner.AUTHORIZED_SEEDS_COUNT * 2} games) "
                               f"[{rate:.2f} seeds/s] - Latest seed {seed}: delta_paired={seed_res['delta_paired']:.2f}")
                    runner.save_results_checkpoint(results_map, start_time_iso, status="IN_PROGRESS")
        
        # Periodic memory reclamation
        gc.collect()

    t_end = time.perf_counter()
    total_elapsed = t_end - t0
    runner.log(f"PROSPECTIVE EXECUTION COMPLETE: {len(results_map)} seeds ({len(results_map)*2} games) in {total_elapsed:.1f}s.")
    runner.save_results_checkpoint(results_map, start_time_iso, status="COMPLETED")

    manifest = {
        "experiment": "EXP-ARCH-002_H1_DOMINANCE_TERMINAL_REEXECUTION",
        "cohort_lo": runner.AUTHORIZED_COHORT_LO,
        "cohort_hi": runner.AUTHORIZED_COHORT_HI,
        "authorized_seeds": runner.AUTHORIZED_SEEDS_COUNT,
        "executed_seeds": len(results_map),
        "total_games": len(results_map) * 2,
        "start_time": start_time_iso,
        "end_time": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "elapsed_seconds": total_elapsed,
        "runner_version": "2.0.0_kaggle_t4x2",
        "runner_sha256": runner.sha256_of_file(runner.__file__),
        "status": "COMPLETED",
    }
    with open(runner.MANIFEST_FILE, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    return results_map

# Run the prospective execution
final_results = execute_with_memory_discipline(workers=WORKERS)


## Section 12: CHECKPOINT MONITORING
Verify checkpoint artifacts on disk and check byte counts.


In [ ]:
# Section 12: CHECKPOINT MONITORING
print("Auditing Checkpoint Artifacts...")
check_files = [
    runner.RAW_RESULTS_FILE,
    runner.PAIRED_DELTAS_FILE,
    runner.TELEMETRY_EVENTS_FILE,
    runner.MANIFEST_FILE,
    runner.LOG_FILE,
]

for cp in check_files:
    p = Path(cp)
    assert p.exists(), f"Missing checkpoint file: {p}"
    print(f"  [OK] {p.name} ({p.stat().st_size:,} bytes)")


## Section 13: POST-EXECUTION INTEGRITY AUDIT
Perform complete scientific integrity verification across all 1,000 games:
- Exactly 500 unique seeds in `[320000, 320499]`
- Exactly 1,000 games
- Zero duplicate seeds
- Zero DEV-007 or Final Holdout leakage
- 100% P-PURE pre-E action identity
- `delta == 0.00` on non-activated games


In [ ]:
# Section 13: POST-EXECUTION INTEGRITY AUDIT
with open(runner.RAW_RESULTS_FILE, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

seeds_data = raw_data["seeds"]
executed_seeds = [s["seed"] for s in seeds_data]

# 1. Seed count and bounds
assert len(executed_seeds) == 500, f"Expected 500 seeds, observed {len(executed_seeds)}"
assert len(set(executed_seeds)) == 500, "Duplicate seeds found!"
assert min(executed_seeds) == 320000, f"Min seed {min(executed_seeds)} != 320000"
assert max(executed_seeds) == 320499, f"Max seed {max(executed_seeds)} != 320499"

# 2. Quarantine verification
for s in executed_seeds:
    assert not (214200 <= s <= 214249), f"CRITICAL BREACH: Holdout seed {s} executed!"
    assert not (310000 <= s <= 310499), f"CRITICAL BREACH: DEV-007 seed {s} executed!"

# 3. P-PURE verification
total_games = 0
pure_pre_e_fails = 0
pure_step_fails = 0
activated_count = 0

for s_entry in seeds_data:
    for g_key in ("game_a", "game_b"):
        g = s_entry[g_key]
        total_games += 1
        if not g["pure_pre_e_pass"]:
            pure_pre_e_fails += 1
        if not g["pure_step_pass"]:
            pure_step_fails += 1
        if g["activated"]:
            activated_count += 1

assert total_games == 1000, f"Expected 1000 games, observed {total_games}"
assert pure_pre_e_fails == 0, f"P-PURE Pre-E Failed on {pure_pre_e_fails} games!"
assert pure_step_fails == 0, f"P-PURE Step Failed on {pure_step_fails} games!"

print("=" * 70)
print("POST-EXECUTION INTEGRITY AUDIT: 100% PASS")
print(f"Total Unique Seeds:  {len(executed_seeds)} (320000 to 320499)")
print(f"Total Games:         {total_games}")
print(f"Holdout Leakage:     0 seeds (100% quarantine maintained)")
print(f"DEV-007 Leakage:     0 seeds (100% quarantine maintained)")
print(f"P-PURE Pre-E Fails:  {pure_pre_e_fails} (100.0% pass)")
print(f"P-PURE Step Fails:   {pure_step_fails} (100.0% pass)")
print(f"Runner State Defect: 0 (ELIMINATED)")
print("=" * 70)


## Section 14: FORENSIC ANALYSIS & STATISTICAL EVALUATION
Analyze activation rates, reason histograms, paired deltas, and statistical criteria according to the preregistered protocol.


In [ ]:
# Section 14: FORENSIC ANALYSIS
import numpy as np
import pandas as pd
from scipy import stats

deltas_df = pd.read_csv(runner.PAIRED_DELTAS_FILE)
telemetry_df = pd.read_csv(runner.TELEMETRY_EVENTS_FILE)

# Activation Statistics
activated_games = int(telemetry_df["activated"].sum())
total_games = len(telemetry_df)
exposed_seeds = int(deltas_df["exposed"].sum())
total_seeds = len(deltas_df)
act_rate = activated_games / total_games

# Exact Clopper-Pearson 95% Confidence Interval for Activation
alpha = 0.05
ci_low = stats.beta.ppf(alpha / 2, activated_games, total_games - activated_games + 1) if activated_games > 0 else 0.0
ci_high = stats.beta.ppf(1 - alpha / 2, activated_games + 1, total_games - activated_games) if activated_games < total_games else 1.0

# Paired Deltas
paired_deltas = deltas_df["delta_paired"].to_numpy()
mean_delta = np.mean(paired_deltas)
std_delta = np.std(paired_deltas, ddof=1)
median_delta = np.median(paired_deltas)
min_delta = np.min(paired_deltas)
max_delta = np.max(paired_deltas)

# Wins / Losses / Ties
wins = int(np.sum(paired_deltas > 0))
losses = int(np.sum(paired_deltas < 0))
ties = int(np.sum(paired_deltas == 0))

# Reason Code Histogram
reasons = telemetry_df["reason"].value_counts().to_dict()

print("=" * 70)
print("EXP-ARCH-002 H1 STATISTICAL EVALUATION")
print("=" * 70)
print(f"Exposed Seeds (k):     {exposed_seeds} / {total_seeds} ({exposed_seeds/total_seeds*100:.2f}%)")
print(f"Activated Games:       {activated_games} / {total_games} ({act_rate*100:.2f}%)")
print(f"Exact 95% CI (Act):    [{ci_low*100:.2f}%, {ci_high*100:.2f}%]")
print(f"Mean Paired Delta:     ${mean_delta:.4f}")
print(f"Std Paired Delta:      ${std_delta:.4f}")
print(f"Median Paired Delta:   ${median_delta:.4f}")
print(f"Delta Range:           [${min_delta:.2f}, ${max_delta:.2f}]")
print(f"Record (W / L / T):    {wins} W / {losses} L / {ties} T")
print("\nReason Code Breakdown:")
for r_code, count in reasons.items():
    print(f"  {r_code:<25}: {count:>5} games ({count/total_games*100:.2f}%)")
print("=" * 70)


## Section 15: OUTPUT ARCHIVE GENERATION
Generate `KAGGLE_EXECUTION_REPORT.md` and pack all artifacts into `ARCH002_KAGGLE_RESULTS_<timestamp>.tar.gz`.


In [ ]:
# Section 15: OUTPUT ARCHIVE GENERATION
import tarfile
import datetime

timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
report_path = OUTPUT_DIR / "KAGGLE_EXECUTION_REPORT.md"

report_md = f"""# EXP-ARCH-002 H1 — KAGGLE EXECUTION & FORENSIC REPORT

**Experiment**: `EXP-ARCH-002_H1_DOMINANCE_TERMINAL`  
**Execution Environment**: Kaggle GPU T4 x 2 Runtime  
**Execution Date**: {timestamp} UTC  
**Status**: **COMPLETED — INTEGRITY VERIFIED**  

---

## 1. Execution Summary
- **Authorized Cohort**: Seeds 320000 to 320499 (N = 500 seeds)
- **Total Games Executed**: 1,000 games (CRN design, 2 games/seed)
- **Quarantine Audit**: Zero holdout or DEV-007 seeds accessed
- **P-PURE Pre-E Action Identity**: 100.0% PASS (Zero runner state pollution)
- **Hardware Target**: Kaggle GPU T4 x 2 (CPU-bound simulator execution on Kaggle vCPUs)

## 2. Statistical Findings
- **Activated Games**: {activated_games} / 1,000 ({act_rate*100:.2f}%)
- **Exposed Seeds**: {exposed_seeds} / 500 ({exposed_seeds/500*100:.2f}%)
- **Mean Paired Delta**: ${mean_delta:.4f}
- **Median Paired Delta**: ${median_delta:.4f}
- **Standard Deviation**: ${std_delta:.4f}
- **Min / Max Delta**: [${min_delta:.2f}, ${max_delta:.2f}]
- **Wins / Losses / Ties**: {wins} W / {losses} L / {ties} T

## 3. Reason Code Breakdown
"""
for r_code, count in reasons.items():
    report_md += f"- `{r_code}`: {count} games ({count/total_games*100:.2f}%)\n"

report_md += """
## 4. Governance Decision & Conclusion
Execution integrity is 100% restored. All runner state pollution has been eliminated.
"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_md)

print(f"Generated report: {report_path}")

# Package archive
archive_name = f"ARCH002_KAGGLE_RESULTS_{timestamp}.tar.gz"
archive_path = Path("/kaggle/working") / archive_name if Path("/kaggle/working").exists() else REPO_ROOT / archive_name

with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(report_path, arcname="KAGGLE_EXECUTION_REPORT.md")
    tar.add(runner.RAW_RESULTS_FILE, arcname="RAW_RESULTS.json")
    tar.add(runner.PAIRED_DELTAS_FILE, arcname="PAIRED_DELTAS.csv")
    tar.add(runner.TELEMETRY_EVENTS_FILE, arcname="TELEMETRY_EVENTS.csv")
    tar.add(runner.MANIFEST_FILE, arcname="EXECUTION_MANIFEST.json")
    tar.add(runner.LOG_FILE, arcname="EXECUTION_LOG.txt")

print("\n" + "=" * 70)
print(f"SUCCESS: Execution archive generated at:")
print(f"  {archive_path} ({archive_path.stat().st_size:,} bytes)")
print("Please download this archive from Kaggle to complete the experiment record.")
print("=" * 70)
